In [5]:
import pandas as pd

# Renaming the columns

I am already doing it now because it's just so ugly huhuhu

In [6]:
# Base columns (the ones before D1...D11)
base_cols = [
    'Sector', 'Subsector', 'Job_Requirement', 'Tasks',
    'Program', 'Type_of_Program', 'Provincial_Rank'
]

# Repeated patterns per Dx
core_pattern = [
    'Priority', 'Next1-3', 'Next1-5', 'Low', 'Medium',
    'High', 'Final_Score', 'Priority_Ranking'
]
irrelevant_pattern = ['Irrelevant1', 'Irrelevant2', 'Irrelevant3']

# Combine everything up to D11
column_names = base_cols + [
    f'D{i}_{p}' for i in range(1, 12) for p in (core_pattern + irrelevant_pattern)
]

column_names = column_names[:-3]

We are skipping jobs where the company did not follow directions like saying a job is low, medium, and high priority all at the same time.

In [7]:
filepath = 'data/ABDD-Phase1B.xlsx'
df = pd.read_excel(filepath, sheet_name=2, skiprows=7, skipfooter=33, names=column_names)
# Drop columns and rows
df = df.drop(df.filter(regex='Irrelevant').columns, axis=1)

# I am dropping these rows because they dont have a lot of missing data
df = df.drop(df.index[range(470, 481)])


In [9]:
def combine_one_hot(df, prefix, order=["Low", "Medium", "High"], as_numeric=False):
    """
    Combine one-hot encoded columns into a single ordinal column.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing the one-hot columns.
        prefix (str): Prefix of the one-hot columns (e.g. 'D1' for 'D1_Low', 'D1_Medium', 'D1_High').
        order (list): Ordered list of category labels.
        as_numeric (bool): If True, returns numeric ranks (1, 2, 3). Else returns labels.

    Returns:
        pd.Series: Combined ordinal column.
    """
    cols = [f"{prefix}_{label}" for label in order]
    # For each row, find which one-hot column has the value 1, take its column name, and 
    # strip off the prefix (like D1_) to get the category label.
    combined = df[cols].idxmax(axis=1).str.replace(f"{prefix}_", "", regex=False)
    
    if as_numeric:
        mapping = {label: i+1 for i, label in enumerate(order)}
        return combined.map(mapping)
    
    return combined

In [10]:
for i in range(1, 12):
    prefix = f"D{i}"
    new_col = f"{prefix}_level"
    df[new_col] = combine_one_hot(df, prefix)

In [21]:
tourism_df = df[df['Sector'].isin(['CONSTRUCTION', 'MANUFACTURING', 'TOURISM (HOTEL AND RESTAURANT)'])]

tourism_df.head()

,Sector,Subsector,Job_Requirement,Tasks,Program,Type_of_Program,Provincial_Rank,D1_Priority,D1_Next1-3,D1_Next1-5,...,D2_level,D3_level,D4_level,D5_level,D6_level,D7_level,D8_level,D9_level,D10_level,D11_level
33,CONSTRUCTION,HORIZONTAL CONSTRUCTION VERTICAL CONSTRUCTION ...,_x000D_\n \t_x000D_\nCARBON STEEL PLATE/ PIPE...,WELD CARBON STEEL PLATES,MANUAL METAL ARC WELDING (MMAW) NC I,With TR,HIGH,True,1.0,True,...,Low,Low,Low,High,Medium,High,Medium,Medium,Medium,Medium
34,CONSTRUCTION,HORIZONTAL CONSTRUCTION VERTICAL CONSTRUCTION ...,_x000D_\n \t_x000D_\nCARBON STEEL PLATE/ PIPE...,WELD AUSTENITICS STAINLESS STEEL PLATE,MANUAL METAL ARC WELDING (MMAW) NC III,With TR,MEDIUM,True,1.0,True,...,Low,Low,Low,Medium,Medium,Medium,Medium,Medium,Medium,Medium
35,CONSTRUCTION,HORIZONTAL CONSTRUCTION VERTICAL CONSTRUCTION ...,_x000D_\n \t_x000D_\nCARBON STEEL PLATE/ PIPE...,WELD AUSTENITIC STAINLESS STEEL PLATES AND PIP...,MANUAL METAL ARC WELDING (MMAW) NC IV,With TR,MEDIUM,True,1.0,True,...,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium,Medium
36,CONSTRUCTION,HORIZONTAL CONSTRUCTION VERTICAL CONSTRUCTION ...,_x000D_\n \t_x000D_\nGAS (OXY-ACETYLENE) WELDER,APPLY THE WELDING TECHNIQUE TO JOIN METAL PIEC...,GAS WELDING NC II,With TR,MEDIUM,True,1.0,True,...,Low,Low,Low,Low,Low,Low,Low,Low,Low,Low
37,CONSTRUCTION,HORIZONTAL CONSTRUCTION VERTICAL CONSTRUCTION,_x000D_\n \t_x000D_\nTRUCK MOUNTED CRANE OPER...,CHECK AND INSPECT TRUCK AND CRANE BEFORE USE\n...,HEAVY EQUIPMENT OPERATION (TRUCK MOUNTED CRANE...,With TR,LOW,True,1.0,True,...,High,Low,Low,Low,Low,Low,Low,Low,Low,Low


In [23]:
levels = [f'D{i}_level' for i in range(1, 12)]
best_mapping = {'High':500, 'Medium':250, 'Low':50}
expected_mapping = {'High':350, 'Medium':175, 'Low':30}
pessimistic_mapping = {'High':250, 'Medium':100, 'Low':10}

best_labor_demand = tourism_df[levels].map(lambda x : best_mapping[x]).sum(axis=0).sum()
expected_labor_demand = tourism_df[levels].map(lambda x : expected_mapping[x]).sum(axis=0).sum()
pessimistic_labor_demand = tourism_df[levels].map(lambda x : pessimistic_mapping[x]).sum(axis=0).sum()

In [26]:
print(f'Best Scenario of Quantity of Labor Demanded: {best_labor_demand}')
print(f'Expected Scenario of Quantity of Labor Demanded: {expected_labor_demand}')
print(f'Pessimistic Scenario of Quantity of Labor Demanded: {pessimistic_labor_demand}')

Best Scenario of Quantity of Labor Demanded: 427300
Expected Scenario of Quantity of Labor Demanded: 292530
Pessimistic Scenario of Quantity of Labor Demanded: 182560


In [41]:
tourism_df.dropna().shape[0] / tourism_df.shape[0]

0.912621359223301